# SCF Phase 2 shard 08

Sweeps: m2, nullcal. Jobs: 4. Projected: 4.4 h
(safety-factor 1.8x applied). Grids hash: `68970a975545`.
Code source: github.com/hugogobato/scf-confounding-frontier @ tag `phase2-freeze` (pinned for
reproducibility).
Pre-registration: `docs/phase2_preregistration.md` (thresholds frozen before
any data generation; deviation register D1-D7 included there).

Resume-safe: completed cells are skipped on rerun (checkpoint parquet per
cell). If the notebook approaches the Colab wall limit it finishes the
current cell and stops cleanly; rerun to continue.

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"
!pip install -q "numpy>=2.0" "scipy>=1.14" "pandas>=2.2" "pyarrow>=16" scikit-learn

In [ ]:
!git clone --depth 1 --branch phase2-freeze \
    https://github.com/hugogobato/scf-confounding-frontier.git scf_repo
import sys, hashlib, json
sys.path.insert(0, "scf_repo/code")
# verify the pinned code matches the manifest recorded at generation time
EXPECTED = json.loads("{\"de_formulas.py\": \"5dffb441b638\", \"simulator.py\": \"ef31ca2a201b\", \"estimators.py\": \"7e27f25b2330\", \"detection.py\": \"06586fe60b9f\", \"runners.py\": \"df67486b60f5\"}")
for fname, short in EXPECTED.items():
    h = hashlib.sha256(open(f"scf_repo/code/{fname}", "rb").read()).hexdigest()[:12]
    assert h == short, f"code mismatch: {fname} ({h} != {short})"
print("code verified against generation-time hashes")

In [ ]:
import json, time, traceback
from multiprocessing import Pool
from runners import run_cell

JOBS = json.loads("[{\"config\": {\"n\": 2000, \"p\": 10000, \"r\": 3, \"l\": [6.708203932499369, 1.118033988749895, 1.118033988749895], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 0.0, \"twin_gamma0\": false, \"q_fixed\": true}, \"config_id\": \"8c0eb115df44\", \"mode\": \"nullcal\", \"sweep\": \"nullcal\", \"reps\": 1200, \"raw_path\": \"data/sim/nullcal/raw/8c0eb115df44.parquet\", \"means_path\": \"data/sim/nullcal/means/8c0eb115df44.npz\", \"_rank\": 1, \"_sweep\": \"nullcal\"}, {\"config\": {\"n\": 2000, \"p\": 4000, \"r\": 5, \"l\": [4.242640687119286, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"m2_weak\", \"m2_treatment\": true, \"m2_tau\": 1.0, \"delta_g\": 0.3, \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"21f32e52c10e\", \"mode\": \"m2\", \"sweep\": \"m2\", \"reps\": 150, \"raw_path\": \"data/sim/m2/raw/21f32e52c10e.parquet\", \"means_path\": \"data/sim/m2/means/21f32e52c10e.npz\", \"_rank\": 7, \"_sweep\": \"m2\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 5, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"m2_weak\", \"m2_treatment\": true, \"m2_tau\": 1.0, \"delta_g\": 0.3, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"1ab9210749fc\", \"mode\": \"m2\", \"sweep\": \"m2\", \"reps\": 150, \"raw_path\": \"data/sim/m2/raw/1ab9210749fc.parquet\", \"means_path\": \"data/sim/m2/means/1ab9210749fc.npz\", \"_rank\": 7, \"_sweep\": \"m2\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"m2_weak\", \"m2_treatment\": true, \"m2_tau\": 1.0, \"delta_g\": 0.3, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"67db7e2b0442\", \"mode\": \"m2\", \"sweep\": \"m2\", \"reps\": 150, \"raw_path\": \"data/sim/m2/raw/67db7e2b0442.parquet\", \"means_path\": \"data/sim/m2/means/67db7e2b0442.npz\", \"_rank\": 7, \"_sweep\": \"m2\"}]")

def _safe(job):
    try:
        return run_cell(job)
    except Exception as e:
        print('[FAIL]', job['config_id'], repr(e))
        traceback.print_exc()
        return job['config_id'], -1.0

t0 = time.time()
results = []
for i, job in enumerate(JOBS):
    if time.time() - t0 > 8.6 * 3600:
        print('[WALL LIMIT] stopping cleanly after', i, 'jobs')
        break
    results.append(_safe(job))
print('shard done:', results)

In [ ]:
import hashlib, json, glob, os
manifest = {'shard_id': 8, 'files': {}}
os.makedirs('data', exist_ok=True)
for f in sorted(glob.glob('data/**/*.parquet', recursive=True)) + \
         sorted(glob.glob('data/**/*.npz', recursive=True)):
    h = hashlib.sha256(open(f, 'rb').read()).hexdigest()
    manifest['files'][f] = h
with open('data/manifest.json', 'w') as fh:
    json.dump(manifest, fh, indent=1)
print(json.dumps(manifest['files'], indent=1))

In [ ]:
import shutil
archive = shutil.make_archive('scf_shard_{:02d}'.format(8), 'zip', 'data')
print('archived:', archive)
output_file = archive
try:
    from google.colab import files
    files.download(output_file)
    print('Downloaded:', output_file)
except Exception as e:
    print('(Not on Colab / download skipped):', e)